# 4 - Guide d'intégration des modèles de différents packages

## Importation des modules

In [1]:
# Importation des modules
# Modules de base
import numpy as np
import pandas as pd
import sys

# Ajout du chemin
sys.path.append('..')

# Importation des utilitaires sklearn
from sklearn.utils.metaestimators import _safe_split
from sklearn.model_selection import cross_val_predict

# Importation des modèles
# Sklearn
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
# XGBoost
from xgboost import XGBRegressor
# Sktime
from sktime.forecasting.arima import ARIMA
# Tslearn

# Darts


# Importation des pipelines
# Sklearn
from sklearn.pipeline import Pipeline

#  Eléments du package à intégrer
# Crossval
from tsforecast.crossvals import (
    PanelOutOfSampleSplit
)

## Création de données synthétiques

In [2]:
# Fonction de génération des données de panel
def generate_panel_data(entities=['A', 'B', 'C'], start_date='2015-01-01', periods=120, 
                        freq='MS', heterogeneous_effects=True, common_trend=True, 
                        entity_specific_seasonality=True, cross_sectional_correlation=0.3,
                        missing_data_prob=0.0):
    """
    Generate synthetic panel data with various characteristics.
    
    Args:
        entities: List of entity identifiers
        start_date: Start date for the panel
        periods: Number of time periods per entity
        freq: Frequency of observations
        heterogeneous_effects: Whether entities have different baseline levels
        common_trend: Whether to include a common trend across entities
        entity_specific_seasonality: Whether seasonality patterns differ by entity
        cross_sectional_correlation: Correlation between entity shocks
        missing_data_prob: Probability of missing observations
    
    Returns:
        pd.DataFrame: Panel data with MultiIndex (entity, date)
    """
    # Création de l'index temporel
    dates = pd.date_range(start=start_date, periods=periods, freq=freq)
    
    # Création du MultiIndex (entity, date)
    index = pd.MultiIndex.from_product([entities, dates], names=['entity', 'date'])
    
    # Initialisation du DataFrame
    panel_data = pd.DataFrame(index=index)
    
    # Génération des effets fixes par entité (hétérogénéité)
    if heterogeneous_effects:
        entity_effects = {entity: np.random.normal(0, 2) for entity in entities}
    else:
        entity_effects = {entity: 0 for entity in entities}
    
    # Tendance commune
    if common_trend:
        common_trend_values = 0.02 * np.arange(periods)
    else:
        common_trend_values = np.zeros(periods)
    
    # Génération de chocs corrélés entre entités
    if cross_sectional_correlation > 0:
        # Chocs communs
        common_shocks = np.random.normal(0, 1, periods)
        # Chocs idiosyncratiques
        idiosyncratic_shocks = {
            entity: np.random.normal(0, 1, periods) 
            for entity in entities
        }
    
    # Construction des séries pour chaque entité
    values = []
    
    # Parcours des entités
    for entity in entities:
        # Effet fixe de l'entité
        entity_effect = entity_effects[entity]
        
        # Saisonnalité spécifique à l'entité
        if entity_specific_seasonality:
            # Période et amplitude différentes selon l'entité
            seasonal_period = 20 + hash(entity) % 40  # Entre 20 et 60
            seasonal_amplitude = 0.5 + (hash(entity) % 100) / 200  # Entre 0.5 et 1.0
        else:
            seasonal_period = 30
            seasonal_amplitude = 0.5
        
        seasonal_values = seasonal_amplitude * np.sin(2 * np.pi * np.arange(periods) / seasonal_period)
        
        # Processus autorégressif spécifique à l'entité
        ar_coef = 0.5 + (hash(entity) % 50) / 100  # Entre 0.5 et 1.0
        ar_process = np.zeros(periods)
        ar_process[0] = np.random.normal(0, 0.5)
        for t in range(1, periods):
            ar_process[t] = ar_coef * ar_process[t-1] + np.random.normal(0, 0.5)
        
        # Combinaison des composantes
        if cross_sectional_correlation > 0:
            # Chocs avec corrélation croisée
            correlated_shocks = (
                np.sqrt(cross_sectional_correlation) * common_shocks +
                np.sqrt(1 - cross_sectional_correlation) * idiosyncratic_shocks[entity]
            )
        else:
            correlated_shocks = np.random.normal(0, 1, periods)
        
        # Combinaison des valeurs
        entity_values = (
            entity_effect + 
            common_trend_values + 
            seasonal_values + 
            ar_process + 
            correlated_shocks
        )
        
        # Ajout de données manquantes
        if missing_data_prob > 0:
            missing_mask = np.random.random(periods) < missing_data_prob
            entity_values[missing_mask] = np.nan
        
        values.extend(entity_values)
    
    # Création du DataFrame final avec les valeurs
    panel_data['value'] = values
    
    # Ajout de variables explicatives
    panel_data['lag_value'] = panel_data.groupby('entity')['value'].shift(1)
    panel_data['trend'] = np.tile(np.arange(periods), len(entities))
    panel_data['month'] = panel_data.index.get_level_values('date').month
    
    return panel_data

In [3]:
# Génération de différents types de données de panel
print("📊 Génération de données de panel ...")

# Panel 1: Données équilibrées avec effets hétérogènes
entities_small = ['FR', 'DE', 'IT', 'ES']
df_panel = generate_panel_data(
    entities=entities_small,
    start_date='2015-01-01',
    periods=120,
    freq='MS',
    heterogeneous_effects=True,
    common_trend=True,
    entity_specific_seasonality=True,
    cross_sectional_correlation=0.4
)

# Suppression des Nan
df_panel.dropna(how='any', inplace=True)

# Séparation en X et y
y = df_panel['value'].copy()
X = df_panel.drop('value', axis=1)

print(f"✅ Génération de panels terminée:")
print(f"Caractéristiques des données générées : {df_panel.shape[0]} observations, {len(entities_small)} entités")

# Affichage des premières observations de chaque panel
print(f"\n📋 Aperçu des données:")
print(df_panel.head(10))

📊 Génération de données de panel ...
✅ Génération de panels terminée:
Caractéristiques des données générées : 476 observations, 4 entités

📋 Aperçu des données:
                      value  lag_value  trend  month
entity date                                         
FR     2015-02-01 -4.525284  -4.919956      1      2
       2015-03-01 -4.508003  -4.525284      2      3
       2015-04-01 -3.643797  -4.508003      3      4
       2015-05-01 -1.956791  -3.643797      4      5
       2015-06-01 -1.591469  -1.956791      5      6
       2015-07-01 -0.676300  -1.591469      6      7
       2015-08-01 -1.966133  -0.676300      7      8
       2015-09-01 -1.162777  -1.966133      8      9
       2015-10-01 -1.062178  -1.162777      9     10
       2015-11-01 -0.794888  -1.062178     10     11


## Intégration de la crossval

### Initialisation de la crossval

In [4]:
# Initialisation de l'horizon de prédiction
horizon=2
# Initialisation du délai de publication
delays=1
# Application de l'horizon aux données afin d'aligner X et y à prévoir
# /!\ Créer une classe plus intelligente qui utilise la régularité de la série (sur données de panel et de séries temporelles) pour ajouter les dates manquantes aux extrémités de la période et ne pas perdre de données
X = X.shift(-horizon)

# Initialisation de la crossval pour l'ensemble des tests
cv = PanelOutOfSampleSplit(
    test_indices=['2024-01-01', '2024-02-01'], 
    test_size=1, 
    gap=horizon + delays
)
splits = list(cv.split(X, y))
# Extraction des indices d'entrainement et de test
train, test = splits[0]

### Intégration des modèles `sklearn`

In [5]:
# Initialisation du modèle
estimator=Ridge()

# Séparation des données d'entrainement et de test
X_train, y_train = _safe_split(estimator, X, y, train)
X_test, _ = _safe_split(estimator, X, y, test, train)
# Entrainement du modèle
estimator.fit(X_train, y_train)
# Prédiction du modèle
y_pred_sklearn = estimator.predict(X_test)

y_pred_sklearn

array([ 2.54850351,  6.49287752, -0.42706331,  6.76984554])

In [6]:
# Initialisation de la pipeline
estimator = Pipeline([
    ('StandardScalerTransformer', StandardScaler()),
    ('RidgeEstimator', Ridge())
])

# Séparation des données d'entrainement et de test
X_train, y_train = _safe_split(estimator, X, y, train)
X_test, _ = _safe_split(estimator, X, y, test, train)
# Entrainement du modèle
estimator.fit(X_train, y_train)
# Prédiction du modèle
y_pred_sklearn_pipeline = estimator.predict(X_test)

y_pred_sklearn_pipeline

array([ 2.54710518,  6.48291411, -0.42200029,  6.7592807 ])

### Intégration des modèles `xgboost`

In [7]:
# Initialisation du modèle
estimator=XGBRegressor()

# Séparation des données d'entrainement et de test
X_train, y_train = _safe_split(estimator, X, y, train)
X_test, _ = _safe_split(estimator, X, y, test, train)
# Entrainement du modèle
estimator.fit(X_train, y_train)
# Prédiction du modèle
y_pred_xgboost = estimator.predict(X_test)

y_pred_xgboost

array([ 3.3325405,  6.386233 , -2.054777 ,  6.6975274], dtype=float32)

In [8]:
# Initialisation de la pipeline
estimator = Pipeline([
    ('StandardScalerTransformer', StandardScaler()),
    ('XGBoostEstimator', XGBRegressor())
])

# Séparation des données d'entrainement et de test
X_train, y_train = _safe_split(estimator, X, y, train)
X_test, _ = _safe_split(estimator, X, y, test, train)
# Entrainement du modèle
estimator.fit(X_train, y_train)
# Prédiction du modèle
y_pred_xgboost_pipeline = estimator.predict(X_test)

y_pred_xgboost_pipeline

array([ 3.3325405,  6.386233 , -2.054777 ,  6.6975274], dtype=float32)

### Intégration des modèles `sktime`

### Intégration des modèles `tslearn`

### Intégration des modèles `darts`